In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score

AttributeError: partially initialized module 'torch' from 'C:\Users\lina ahmed\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\__init__.py' has no attribute 'fx' (most likely due to a circular import)

In [ ]:
DATA_DIR = "data/processed"
MODEL_DIR = "models/deep_learning"
os.makedirs(MODEL_DIR, exist_ok=True)
 
RANDOM_SEED = 42
BATCH_SIZE = 256
MAX_EPOCHS = 100
PATIENCE = 8            # early stopping patience (epochs w/o val PR-AUC improvement)
LEARNING_RATE = 1e-3
 
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

train_df = pd.read_csv(f"{DATA_DIR}/model_train.csv")
val_df = pd.read_csv(f"{DATA_DIR}/model_val.csv")
test_df = pd.read_csv(f"{DATA_DIR}/model_test.csv")


In [ ]:
numeric_features = train_df.drop(columns=["Class"]).select_dtypes(include=[np.number]).columns.tolist()
categorical_features = train_df.drop(columns=["Class"]).select_dtypes(include=[object]).columns.tolist()
 
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
 
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])
 
# Fit ONLY on train, transform all three splits (no leakage)
X_train = preprocessor.fit_transform(train_df.drop(columns=["Class"])).astype(np.float32)
X_val = preprocessor.transform(val_df.drop(columns=["Class"])).astype(np.float32)
X_test = preprocessor.transform(test_df.drop(columns=["Class"])).astype(np.float32)
 
y_train = train_df["Class"].to_numpy().astype(np.float32)
y_val = val_df["Class"].to_numpy().astype(np.float32)
y_test = test_df["Class"].to_numpy().astype(np.float32)

joblib.dump(preprocessor, f"{MODEL_DIR}/mlp_preprocessor.pkl")

In [ ]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
# weight the fraud class inversely to its frequency, same idea as pos_weight in the torch version
class_weight = {0: 1.0, 1: float(n_neg / n_pos)}
print("class_weight:", class_weight)
 
def build_model(input_dim, hidden_dims=(64, 32, 16), dropout=0.3):
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for h in hidden_dims:
        x = keras.layers.Dense(h, activation="relu")(x)
        x = keras.layers.Dropout(dropout)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs)
 
model = build_model(input_dim=X_train.shape[1])
 
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(curve="PR", name="pr_auc")],
)
 
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_pr_auc", mode="max", patience=PATIENCE, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_pr_auc", mode="max", factor=0.5, patience=3
    ),
]


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
val_probs = model.predict(X_val, batch_size=BATCH_SIZE).ravel()
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
 
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores[:-1])  # last point has no corresponding threshold
best_threshold = float(thresholds[best_idx])
print(f"Chosen threshold (best val F1): {best_threshold:.4f}")
 
test_probs = model.predict(X_test, batch_size=BATCH_SIZE).ravel()
test_preds = (test_probs >= best_threshold).astype(int)
 
test_pr_auc = average_precision_score(y_test, test_probs)
test_f1 = f1_score(y_test, test_preds)
test_recall = (test_preds[y_test == 1] == 1).mean()
test_precision = (y_test[test_preds == 1] == 1).mean()
 
print(f"Test PR-AUC: {test_pr_auc:.4f}")
print(f"Test F1 @ chosen threshold: {test_f1:.4f}")


In [ ]:
model.save(f"{MODEL_DIR}/mlp_model.keras")
with open(f"{MODEL_DIR}/mlp_threshold.txt", "w") as f:
    f.write(str(best_threshold))
 
mlp_result = pd.DataFrame({
    "Model": ["MLP"],
    "Recall": [test_recall],
    "Precision": [test_precision],
    "F1 Score": [test_f1],
    "PR AUC": [test_pr_auc],
})
mlp_result.to_csv(f"{MODEL_DIR}/mlp_results.csv", index=False)
print(mlp_result)
